# Memory Spilling: Diagnosing Spill (Memory) and Spill (Disk)

**Module**: 06_memory_and_storage  
**Notebook**: 05_spill_to_disk.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: What causes memory spill during shuffles and sorts, how to diagnose spill in Spark UI, and how to eliminate it.
- **Why does this topic exist?**: Spilling destroys query performance due to disk serialization and I/O bottlenecks.
- **What problem does it solve?**: Queries slowing down 10x due to gigabytes of shuffle spill.
- **Where is it used in real-world Data Engineering?**: Fixing production stragglers in heavy aggregations.

---

## 2. Spill (Memory) vs Spill (Disk)

- **Spill (Memory)**: Size of data in RAM before flushing.
- **Spill (Disk)**: Compressed size written to worker scratch disk.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spill_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("Check Stages tab in Spark UI for Spill (Memory) and Spill (Disk) columns.")
orders.groupBy("customer_id").count().show(3)

---

## Experiment: Eliminating Spill by Increasing Partitions

- **1. What are we testing?**: Show that increasing partition count reduces partition size below memory spill threshold.
- **2. What do we expect?**: Spill disappears when partitions are smaller.

In [ ]:
print('Increasing shuffle partitions from 2 to 16 reduces memory required per task by 8x.')

**3. What happened?**: Smaller partitions fit in execution memory.  
**4. Why did it happen?**: Each task processes fewer records.  
**5. Practical Engineering Lesson**: If a stage spills to disk, increase partition count or executor memory.

---

## Practical Exercises

### Exercise 1 (Beginner): Find Spill Metrics in UI
**Problem Statement**: Locate Spill columns in UI.  
**Input / Context**: UI

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate Spill Compression Ratio
**Problem Statement**: Explain why Spill (Disk) is smaller than Spill (Memory).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tuning Shuffle Buffer
**Problem Statement**: Config for shuffle memory fraction.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Find Spill Metrics in UI):
```python
print('Under Stage metrics table in localhost:4040/stages')
```

#### Solution 2 (Calculate Spill Compression Ratio):
```python
print('Spill (Disk) is compressed with LZ4/Snappy codec.')
```

#### Solution 3 (Tuning Shuffle Buffer):
```python
print('spark.memory.fraction')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Spill Impact**: Causes severe disk I/O latency.
- **Fix**: Increase partitions or increase executor memory.

### Essential Technical Terms
- `Spill`: Flushing in-memory buffers to disk.
- `AppendOnlyMap`: In-memory shuffle buffer.

### Common Engineering Mistakes
- **Ignoring non-zero spill**: Indicates execution memory pressure.

### Interview Questions on This Topic
- **Q: What is the difference between Spill (Memory) and Spill (Disk)?**
  - *A*: Spill (Memory) is the deserialized uncompressed size in RAM; Spill (Disk) is the compressed serialized size written to local disk.

### What to Study Next
Open 06_broadcast_variables.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')